In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os,sys,re

!pip install einops -q

if not os.path.isdir("/content/FCCNs"):
  !git clone https://github.com/saurabhya/FCCNs.git /content/FCCNs
else:
  print("FCCNs already cloned")

path = '/content/FCCNs/utils.py'
code = open(path).read()
open(path, 'w').write(re.sub(r'_, term_width = os\.popen.*', 'term_width = 80', code))
sys.path.insert(0, '/content/FCCNs')
os.chdir('/content/FCCNs')

print('✅ Setup complete')

import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms as tf
import networks as models
from utils import ToHSV, ToComplex
from google.colab import files as colab_files

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# from google.colab import drive
# drive.mount('/content/drive')

m_path = '/content/drive/MyDrive/Copy of resnet152_model_best.pth.tar'

model = models.resnet152(num_classes=1000)
checkpoint = torch.load(m_path, map_location=DEVICE)
model.load_state_dict(checkpoint['state_dict'],strict=False)
model=model.to(DEVICE)
model.eval()

class FCCNFeatureExtraction(torch.nn.Module):
  def __init__(self,full_model):
    super().__init__()
    self.conv1=full_model.conv1
    self.bn1=full_model.bn1
    self.prelu=full_model.prelu
    self.maxpool=full_model.maxpool
    self.layer1=full_model.layer1
    self.layer2=full_model.layer2
    self.layer3=full_model.layer3
    self.layer4=full_model.layer4
    self.avgpool=full_model.avgpool

  def forward(self,x):
    x=self.conv1(x)
    x=self.bn1(x)
    x=self.prelu(x)
    x=self.maxpool(x)
    x=self.layer1(x)
    x=self.layer2(x)
    x=self.layer3(x)
    x=self.layer4(x)
    x=self.avgpool(x)
    x=x.squeeze(-1).squeeze(-1)
    return x

# Now each image is represented as: 512-number feature vector
# instead of : 1000 classes
extractor = FCCNFeatureExtraction(model).to(DEVICE)
extractor.eval()

transform = tf.Compose([
    tf.Resize((224,224)),
    tf.ToTensor(),
    ToHSV(),
    ToComplex()
])

def feature_extractor(image):
  x = transform(image).unsqueeze(0).to(DEVICE) #unsqueeze(0) adds batch dimension i.e, converts (3,224,224) to (1,3,224,224), as nn expects batches.
  features = extractor(x)
  features = features.squeeze(0).cpu() #(1,512) → (512,)
  mag = features.abs().detach().numpy()
  phase = features.angle().detach().numpy()
  mag_phase = np.concatenate([mag,phase])
  real_img_concatenate = np.concatenate([features.real.detach().numpy(), features.imag.detach().numpy()])
  return {
          'complex_vec':features.detach().numpy(),
          'mag'     : mag,
          'phase'         : phase,
          'real_img_cat' : real_img_concatenate,
          'mag_phase': mag_phase,
          'stats': {
              'mean'   : float(mag.mean()),
              'std'    : float(mag.std()),
              'min'    : float(mag.min()),
              'max'    : float(mag.max()),
              'active' : int((mag > 1e-4).sum()),
          }
  }

# from google.colab import files
# from PIL import Image
# import io

# uploaded = files.upload()

# filename = list(uploaded.keys())[0]

# result = feature_extractor(io.BytesIO(uploaded[filename]))
# plt.imshow(Image.open(filename))
# result

FCCNs already cloned
✅ Setup complete


In [ ]:
import os
from PIL import Image
train_file_path = "/content/drive/MyDrive/NewR22/train"
test_file_path = "/content/drive/MyDrive/NewR22/test"

def load_images(path):
  images,labels =[],[]
  class_map ={'Normal':0,'Defective':1}
  for class_name,label in class_map.items():
    folder = os.path.join(path,class_name)

    for filename in os.listdir(folder):
      filepath = os.path.join(folder,filename)
      try:
        image = Image.open(filepath).convert('RGB')
        images.append(image)
        labels.append(label)
      except Exception as e:
        print(f'Skipping {filename}: {e}')


  return images,labels

train_images,train_labels = load_images(train_file_path)
test_images,test_labels = load_images(test_file_path)

In [ ]:
print(f'Train: {len(train_images)} images loaded')
print(f'Test:  {len(test_images)} images loaded')

print(f'Train — Normal: {train_labels.count(0)}, Defective: {train_labels.count(1)}')
print(f'Test  — Normal: {test_labels.count(0)}, Defective: {test_labels.count(1)}')

Train: 444 images loaded
Test:  111 images loaded
Train — Normal: 223, Defective: 221
Test  — Normal: 56, Defective: 55


In [ ]:
train_mag,train_real_imag,train_mag_phase,train_labels_arr=[],[],[],[]
for img,label in zip(train_images,train_labels):
  features = feature_extractor(img)
  train_mag.append(features['mag'])
  train_real_imag.append(features['real_img_cat'])
  train_mag_phase.append(features['mag_phase'])
  train_labels_arr.append(label)
train_labels_arr = np.array(train_labels_arr)
train_mag = np.array(train_mag)
train_real_imag = np.array(train_real_imag)
train_mag_phase = np.array(train_mag_phase)

In [ ]:
test_mag, test_real_imag, test_mag_phase, test_labels_arr = [], [], [], []

for img, label in zip(test_images, test_labels):

    features = feature_extractor(img)

    test_mag.append(features['mag'])

    test_real_imag.append(features['real_img_cat'])

    test_mag_phase.append(features['mag_phase'])

    test_labels_arr.append(label)

test_labels_arr = np.array(test_labels_arr)

test_mag = np.array(test_mag)

test_real_imag = np.array(test_real_imag)

test_mag_phase = np.array(test_mag_phase)

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix,ConfusionMatrixDisplay,recall_score

representations = {
    'Magnitude'  : (train_mag,       test_mag),
    'Real+Imag'  : (train_real_imag, test_real_imag),
    'Mag+Phase'  : (train_mag_phase, test_mag_phase),
}

classifiers ={
    'SVC':SVC(kernel='rbf',C=1),
    'RandomForest':RandomForestClassifier(n_estimators=100),
    'LogisticRegression':LogisticRegression(max_iter=2000),
    'KNeighbors':KNeighborsClassifier(n_neighbors=5)
}

import pandas as pd

# you already have all your results — store them in a list as you compute
rows = []

for rep_name, (train_X, test_X) in representations.items():
    for clf_name, clf in classifiers.items():
        clf.fit(train_X, train_labels_arr)
        preds  = clf.predict(test_X)
        acc    = accuracy_score(test_labels_arr, preds)
        f1     = f1_score(test_labels_arr, preds)
        recall = recall_score(test_labels_arr, preds)

        rows.append({
            'Representation' : rep_name,
            'Classifier'     : clf_name,
            'Accuracy'       : round(acc, 4),
            'F1'             : round(f1, 4),
            'Recall'         : round(recall, 4),
        })

# build dataframe
df = pd.DataFrame(rows)
print(df.to_string(index=False))

Representation         Classifier  Accuracy     F1  Recall
     Magnitude                SVC    0.9279 0.9231  0.8727
     Magnitude       RandomForest    0.8919 0.8868  0.8545
     Magnitude LogisticRegression    0.9369 0.9358  0.9273
     Magnitude         KNeighbors    0.8198 0.7872  0.6727
     Real+Imag                SVC    0.9099 0.9038  0.8545
     Real+Imag       RandomForest    0.9009 0.8972  0.8727
     Real+Imag LogisticRegression    0.9550 0.9541  0.9455
     Real+Imag         KNeighbors    0.8198 0.7872  0.6727
     Mag+Phase                SVC    0.9279 0.9245  0.8909
     Mag+Phase       RandomForest    0.9009 0.8952  0.8545
     Mag+Phase LogisticRegression    0.9459 0.9455  0.9455
     Mag+Phase         KNeighbors    0.7387 0.6420  0.4727


In [ ]:
import torchvision.models as real_models
from torchvision import transforms as real_transform

real_model = real_models.resnet152(pretrained=True)
real_model.eval()
real_model = real_model.to(DEVICE)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /root/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth


100%|██████████| 230M/230M [00:06<00:00, 39.1MB/s]


In [ ]:
standard_transform = real_transform.Compose([
    real_transform.Resize((224, 224)),
    real_transform.ToTensor(),
    real_transform.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
class RealResNetExtractor(torch.nn.Module):
    def __init__(self, full_model):
        super().__init__()
        # resnet152 layers
        self.conv1   = full_model.conv1
        self.bn1     = full_model.bn1
        self.relu    = full_model.relu      # ← ReLU not CPReLU
        self.maxpool = full_model.maxpool
        self.layer1  = full_model.layer1
        self.layer2  = full_model.layer2
        self.layer3  = full_model.layer3
        self.layer4  = full_model.layer4
        self.avgpool = full_model.avgpool
        # fc excluded

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.squeeze(-1).squeeze(-1)  # (B, 2048) real
        return x

real_extractor = RealResNetExtractor(real_model).to(DEVICE)
real_extractor.eval()

RealResNetExtractor(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1)

In [ ]:
def real_feature_extractor(image):
    x = standard_transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        features = real_extractor(x)
    features = features.squeeze(0).cpu().numpy()
    return features

In [ ]:
real_train_features = []
real_train_labels=[]

for img, label in zip(train_images, train_labels):
    feat = real_feature_extractor(img)
    real_train_features.append(feat)
    real_train_labels.append(label)

real_train_features = np.array(real_train_features)
print(real_train_features.shape)

(444, 2048)


In [ ]:
real_test_features = []
real_test_labels=[]

for img, label in zip(test_images, test_labels):
    feat = real_feature_extractor(img)
    real_test_features.append(feat)
    real_test_labels.append(label)

real_test_features = np.array(real_test_features)
print(real_test_features.shape)

(111, 2048)


In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix,ConfusionMatrixDisplay,recall_score

representations = {
    'Magnitude'  : (real_train_features, real_test_features)
}

classifiers ={
    'SVC':SVC(kernel='rbf',C=1),
    'RandomForest':RandomForestClassifier(n_estimators=100),
    'LogisticRegression':LogisticRegression(max_iter=2000),
    'KNeighbors':KNeighborsClassifier(n_neighbors=5)
}

import pandas as pd

# you already have all your results — store them in a list as you compute
rows = []

for rep_name, (train_X, test_X) in representations.items():
    for clf_name, clf in classifiers.items():
        clf.fit(train_X, train_labels_arr)
        preds  = clf.predict(test_X)
        acc    = accuracy_score(test_labels_arr, preds)
        f1     = f1_score(test_labels_arr, preds)
        recall = recall_score(test_labels_arr, preds)

        rows.append({
            'Representation' : rep_name,
            'Classifier'     : clf_name,

            'Accuracy'       : round(acc, 4),
            'F1'             : round(f1, 4),
            'Recall'         : round(recall, 4),
        })

# build dataframe
df = pd.DataFrame(rows)
print(df.to_string(index=False))

Representation         Classifier  Accuracy     F1  Recall
     Magnitude                SVC    0.9099 0.9057  0.8727
     Magnitude       RandomForest    0.9099 0.9057  0.8727
     Magnitude LogisticRegression    0.9099 0.9091  0.9091
     Magnitude         KNeighbors    0.8739 0.8704  0.8545
